# Lesson 4 — Noul, and picking the right primitive

Lesson 3 covered `Score`. This lesson covers `Noul` — the simplest primitive — and closes out the primitives with a straight comparison: how to tell which of the three a given judgment actually needs.

Ask your agent if anything's unclear.

<details>
  <summary><strong>Primary source:</strong> docs.typesafe.ai/primitives/noul (click to expand)</summary>

# Noul

> A Noul question asks the model to evaluate a yes/no question and return the probability that the answer is yes.

Use a Noul when the answer is yes or no. For example, does this message ask for a refund, does this resume mention distributed systems, does this comment contain personal data. If the answer is one of several options, use a [Choice](https://docs.typesafe.ai/primitives/choice). If it's a position on a spectrum, use a [Score](https://docs.typesafe.ai/primitives/score). [Choose a question type](https://docs.typesafe.ai/primitives#choose-a-question-type) compares all three.

A Noul answer is a single number, `noul`, the probability that the answer is yes.

## Writing a Noul question

A Noul question evaluates a single yes/no question (or statement). It is defined by its `instructions`: the yes/no question to evaluate. It's good practice to phrase it so a high probability means "yes", so that the returned answer is unambiguous in its meaning.

You can optionally add `criteria` with `true` and `false` descriptions to clarify what each outcome means, which can be helpful when the question itself has more nuance to explain. Try your Noul question prompts with and without criteria to see which works better in your use-case.

## Request

| Field | Required | Description |
| --- | --- | --- |
| `type` | Yes | Must be `"noul"`. |
| `instructions` | Yes | The yes/no question or statement to evaluate. |
| `criteria` | No | Optional `{ true, false }` descriptions clarifying what a yes and a no mean. |

```json
{
  "state": "I have asked three times now. Can I please just talk to a real person?",
  "questions": {
    "is_human_escalation": {
      "type": "noul",
      "instructions": "Is the customer asking for a human agent?"
    },
    "is_repeat_contact": {
      "type": "noul",
      "instructions": "Has the customer contacted support about this before?",
      "criteria": {
        "true": "Mentions a prior attempt, ticket, or that they have asked before",
        "false": "No sign of any previous contact"
      }
    }
  }
}
```

## Response

```json
{
  "model": "jev-latest",
  "answers": {
    "is_human_escalation": {
      "type": "noul",
      "noul": 0.99
    },
    "is_repeat_contact": {
      "type": "noul",
      "noul": 0.93
    }
  },
  "usage": {
    "input_tokens": 360,
    "output_tokens": 39
  }
}
```

`noul` ranges from 0 to 1, representing the probability that the answer is **yes**. Most often you will threshold it into a boolean when your code needs a hard decision.

## Noul does not return a separate confidence value

A value near 1 means a strong yes. A value near 0 means a strong no. A value near 0.5 gives yes and no similar probability.

For "Is the candidate strong in Python?", define what "strong" means. An unclear definition makes the probability hard to interpret. A value of 0.5 does not mean medium skill. Use a [Score](https://docs.typesafe.ai/primitives/score) to measure skill along defined levels.

## Example questions

```
"Is the customer requesting a refund?"
"Does this resume mention experience with distributed systems?"
"Does the message contain personally identifiable information?"
"Does the room have a minifridge?"
```

## Tips and advanced usage

- **Phrasing.** Beyond a plain question, you can phrase the instruction as a statement for the model to evaluate for truthfulness. For "the customer is requesting a refund", a value near 1 means the statement is true. Try both phrasings with your own data to see what works best.
- **Optional `criteria`.** The instruction is enough for most Noul questions, but when the boundary between yes and no is subtle, pass `criteria` with `true` and `false` descriptions to pin down what each outcome means — as shown in the request example above.

</details>

## Noul, in your own words

`Noul` is the simplest primitive: one yes/no question, one number back. No options to define, no levels to write — just `noul`, the probability the answer is yes.

The one thing worth getting right: **Noul has no separate `confidence` field.** `0.5` isn't "medium confidence" — it means the model gives yes and no equal probability. If you find yourself wanting a "how sure are you" reading alongside a yes/no answer, that's usually a sign the question is really a `Score` in disguise.

In [ ]:
import { client } from "../../assets/typesafe-client.ts";
import { noul } from "npm:@typesafe-ai/sdk@^0.7.0";

const response = await client.systemOne({
  state: "I have asked three times now. Can I please just talk to a real person?",
  questions: {
    isHumanEscalation: noul("Is the customer asking for a human agent?"),
    isRepeatContact: noul(
      "Has the customer contacted support about this before?",
      {
        true: "Mentions a prior attempt, ticket, or that they have asked before",
        false: "No sign of any previous contact",
      },
    ),
  },
});

console.log(response.answers.isHumanEscalation);
console.log(response.answers.isRepeatContact);

## What just happened

Two independent yes/no judgments against the same state, each returning just a `noul` value — no `confidence`, no `probabilities`, no `legend`. `isHumanEscalation` should land close to 1 (the message is a direct, explicit ask); `isRepeatContact` is more interesting, since the model has to infer "three times now" implies prior contact without being told that directly.

**Try it yourself:** rewrite the message so it's genuinely ambiguous about whether this is a repeat contact (remove "three times now" entirely). Predict whether `isRepeatContact` moves toward 0.5 before you re-run.

<details>
  <summary><strong>Primary source (excerpt):</strong> docs.typesafe.ai/primitives — "Choose a question type" (click to expand)</summary>

| Type | What it answers | Returns |
| --- | --- | --- |
| [Choice](https://docs.typesafe.ai/primitives/choice) | Which of these options? | `choice`, `probabilities`, `confidence` |
| [Score](https://docs.typesafe.ai/primitives/score) | Which level? | `score`, `legend`, `probabilities`, `confidence` |
| [Noul](https://docs.typesafe.ai/primitives/noul) | Is this true? | `noul` (0 to 1) |

## Choose a question type

Pick the type that matches the shape of the answer you need.

- **Choice** fits when the answer is one of a known set of options with no order between them: routing a ticket to a department, classifying a document type, detecting a programming language. Give the full list of options, and add an `other` or `none of the above` option when the list might not cover every input.
- **Score** fits when the answer falls on a spectrum and you can describe what each point on that spectrum means: bug severity, customer frustration, skill level. The levels are yours to define, and the model returns a position along them.
- **Noul** fits a clean yes/no question where the probability itself is the useful signal: does this message report a bug, is the customer requesting a refund, does the resume mention distributed systems.

> **Note:** Use Noul for a yes/no judgment and Score to measure a position on a spectrum. "Is this candidate strong in Python?" needs a clear definition of "strong". A Noul value of 0.5 means the model gives yes and no equal probability. It does not mean the candidate has a medium skill level. An unclear definition makes that probability hard to interpret.
>
> If you want to measure skill level, use a Score with defined levels, such as no experience, some familiarity, daily use, and deep expertise. If you need a yes/no decision, define the condition clearly, such as "Does the resume state that the candidate has used Python at work?"

If two types both seem to fit, prefer the one whose answer your code can act on directly. A Choice between `refund`, `rebook`, and `information` maps straight onto three code paths. A Score of customer frustration maps onto a threshold. A Noul maps onto an `if`.

</details>

## Why picking the right one matters

The three primitives aren't interchangeable skins on the same idea — they shape what your code can do with the answer:

- **Choice** → a fixed set of code paths (`switch` on the result)
- **Score** → a position you threshold or rank against (`if score > 1.5`)
- **Noul** → a boolean decision (`if isRefundRequest`)

Picking the wrong one costs you downstream. A `Noul` where you actually needed a `Score` throws away the "how much" signal and leaves you re-deriving it from a single probability that was never designed to carry it. A `Choice` where you needed a `Score` forces the model to pick a discrete bucket instead of reporting a position that could legitimately sit between two of your buckets.

## Next up

Next lesson: Structure — `instructions` and `criteria` don't have to be plain strings. All three primitives can take structured JSON instead, which matters once your questions get more specific than "what category is this."

Questions? Ask your agent — it has the same research this lesson was built from, in `research/typesafe-ai.md`.